# Assignment Sesi 28 Tugas 1
Nama: Faraday Barr Fatahillah

Dataset: [movies_metadata.csv](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv) 

**Tugas 1**

Pada tugas ini gunakanlah dataset film yang dapat diunduh dari Kaggle [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv). Gunakan file `movies_metadata.csv` sebagai sumber data.Dari data tersebut kerjakanlah beberapa perintah berikut:
1. Buatlah keyword Search menggunakan BM25 dan TF-IDF untuk mencari film berkaitan dengan:
    - robot
    - love
    - adventure
2. Buatlah Semantic Search dengan Vector Database (Pinecone dan ChromaDB) untuk mencari film berkaitan dengan:
    - Adventure movie with characters named Judy and Peter
    - Movies with scenes in New York City
    - movie about poet Arthur Rimbaud and Paul Verlaine relationship"

In [ ]:
# import needed library
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import numpy as np
import chromadb
import google.generativeai as genai
import time
from pinecone import Pinecone, ServerlessSpec

In [ ]:
df = pd.read_csv('movies_metadata.csv')

df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df = df[['title', 'overview']].dropna()

df.isna().sum().sort_values(ascending=False)

## Keyword Search

### TF-IDF Search

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix  = tfidf.fit_transform(df['overview'])

In [ ]:
def format_output(df, top_indices):
    return {
            i: {
                "title": df['title'].iloc[i],
                "overview": df['overview'].iloc[i]
            }
            for i in top_indices
        }




def tfidf_search(query, top_n=10):
    query_vec = tfidf.transform([query])
    cosine_sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = cosine_sim.argsort()[-top_n:][::-1]
    return format_output(df, top_indices)

In [ ]:
df_robot = pd.DataFrame(tfidf_search("robot")).T.reset_index(drop=True)
df_love = pd.DataFrame(tfidf_search("love")).T.reset_index(drop=True)
df_adventure = pd.DataFrame(tfidf_search("adventure")).T.reset_index(drop=True)


In [ ]:
df_robot

In [ ]:
df_love

In [ ]:
df_adventure

### BM25

In [ ]:
tokenized_corpus = [doc.split(" ") for doc in df['overview']]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_n=10):
    tokenized_query = query.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[-top_n:][::-1]
    return format_output(df, top_indices)

In [ ]:
df_robot_bm = pd.DataFrame(tfidf_search("robot")).T.reset_index(drop=True)
df_love_bm = pd.DataFrame(tfidf_search("love")).T.reset_index(drop=True)
df_adventure_bm = pd.DataFrame(tfidf_search("adventure")).T.reset_index(drop=True)


In [ ]:
df_robot_bm

In [ ]:
df_love_bm

In [ ]:
df_adventure_bm

## Semantic Search

### ChromaDB

In [ ]:
genai.configure(api_key='GEMINI_API_KEY')

In [ ]:
def get_embedding(text, retries=5):
    for i in range(retries):
        try:
            result = genai.embed_content(
                model="gemini-embedding-001",
                content=text
            )["embedding"]
            return result
        except Exception as e:
            wait = 2 ** i * 5
            print(f"Rate limit hit, waiting {wait}s... ({e})")
            time.sleep(wait)
    return None

In [ ]:
embeddings = []
for i, text in enumerate(df['overview']):
    emb = get_embedding(text)
    embeddings.append(emb)
    
    time.sleep(1.5)
    
    if (i + 1) % 50 == 0:
        print(f"Progress: {i+1}/{len(df)} rows done")
        time.sleep(10)

In [ ]:
df['embedding'] = embeddings
print("Done!")

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name="movies")

In [ ]:
for i, row in df.iterrows():
    collection.add(
        documents=[row['overview']],
        embeddings=[row['embedding']],
        metadatas=[{"title": row['title']}],
        ids=[str(i)]
    )

In [ ]:
def chroma_search(query, top_n=5):
    query_embedding = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_n
    )
    
    titles = results['metadatas'][0]
    docs = results['documents'][0]
    for i, (meta, doc) in enumerate(zip(titles, docs)):
        print(f"{i+1}. {meta['title']}")
        print(f"   {doc[:150]}...")
        print()
    
    return results

In [ ]:
print("=== Query 1 ===")
chroma_search("Adventure movie with characters named Judy and Peter")

In [ ]:
print("=== Query 2 ===")
chroma_search("Movies with scenes in New York City")

In [ ]:
print("=== Query 3 ===")
chroma_search("movie about poet Arthur Rimbaud and Paul Verlaine relationship")

### Pinecone

In [ ]:
pc = Pinecone(api_key='PINECONE_API_KEY')

index_name = "movies-semantic"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
batch_size = 100
vectors = []

for i, row in df.iterrows():
    vectors.append({
        "id": str(i),
        "values": row['embedding'],
        "metadata": {
            "title": row['title'],
            "overview": row['overview'][:500]
        }
    })
    
    if len(vectors) == batch_size:
        index.upsert(vectors=vectors)
        vectors = []

if vectors:
    index.upsert(vectors=vectors)

In [ ]:
def pinecone_search(query, top_n=5):
    query_embedding = get_embedding(query)
    
    results = index.query(
        vector=query_embedding,
        top_k=top_n,
        include_metadata=True
    )
    
    for i, match in enumerate(results['matches']):
        print(f"{i+1}. {match['metadata']['title']} (score: {match['score']:.4f})")
        print(f"   {match['metadata']['overview'][:150]}...")
        print()
    
    return results

In [ ]:
print("=== Query 1 ===")
pinecone_search("Adventure movie with characters named Judy and Peter")

In [ ]:
print("=== Query 2 ===")
pinecone_search("Movies with scenes in New York City")

In [ ]:
print("=== Query 3 ===")
pinecone_search("movie about poet Arthur Rimbaud and Paul Verlaine relationship")